### 遷移金属酸化物に対する協調フィルタリング


構造データベースから
例えば、InとPからなる二元物質を検索すると

InP　F-43m
InP3 　	R-3m
InP  	Fm-3m
などの構造が見つかります。

これは

|formula| element1| element2 | ratio1 | ratio2 | spacegroup|
|---|---|---|---|---|---|
|InP|In|P|1|1|F-43m|
|InP3|In|P|1|3|R-3m|
|InP|In|P|1|1|fm-3m|

という存在データがあるとみなせます。
このデータを用いて他の物質が存在するかの予測が行えるでしょうか。
この問題に対して協調フィルタリングを用いることができます。



{Al, Si, P, Ga, Ge, As, In, Sn, Sb}から成る
二元物質をCrystallography Open Database (http://www.crystallography.net/cod/)
から得たデータを用います。

次元圧縮をSVDとNMFを用いて行うために更に二次元行列に変換します。
行を"element1^element2"、列を"ratio1^ratio2^spacegroup"という表記を用いて該当物質がデータベースに存在したら＝１、無い場合は＝０とします。

上のような表をすでに用意しています。

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 60)


def get_data():
    df = pd.read_csv("../data/group131415_div1.csv", index_col=[0])
    print(df.shape)
    return df


g_df = get_data()


可視化コードで存在している物質を白く表示します。


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline


def plot_df_heatmap(df, filename=None):
    """plot headmap of df data

    Args:
        df (pd.DataFrame): data
        filename (str, optional): filename. Defaults to None.
    """
    size = np.array(df.shape)[::-1]*0.1
    fig, ax = plt.subplots(figsize=size, dpi=120)
    sns.set(font_scale=0.6)
    sns.heatmap(df, lw=0.1, ax=ax)
    plt.tight_layout()
    if filename is not None:
        fig.savefig(filename)


plot_df_heatmap(g_df)


更に可視化コードを作成しておきます。

In [ ]:
def plot_2df(df, df_transform, nrank):
    """plot the original data and reconstructed data

    Args:
        df (pd.DataFrame): data
        df_transform (pd.DataFrame): reconstructed data
        nrank (int): rank
    """
    figsize = np.array(df.shape).astype(float)[::-1]*0.2
    figsize[0] = figsize[0]*1.8
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    ax = axes[0]
    sns.heatmap(df, ax=ax)
    ax = axes[1]
    ax.set_title("nrank={}".format(nrank))
    sns.heatmap(df_transform, lw=0.1, ax=ax)
    fig.tight_layout()
    fig.show()


In [ ]:
def plot_svd_sdiag(X):
    """寄与率の表示

    Args:
        X (np.array): descriptor
    """
    u, sdiag, v = np.linalg.svd(X)
    if True:
        n = sdiag.shape[0]
        s = np.zeros((u.shape[1], v.shape[0]))
        s[:n, :n] = np.diag(sdiag)
        u = np.matrix(u)
        v = np.matrix(v)  # = v.T
        s = np.matrix(s)
        usv = u*s*v
        print("check usv = original matrix? ", np.allclose(usv, X))

    sdiagsum = []
    for i in range(sdiag.shape[0]):
        sdiagsum.append(np.sum(sdiag[:i+1]))
    sdiagsum = np.array(sdiagsum)
    sdiag = sdiag / sdiagsum[-1]
    sdiagsum = sdiagsum / sdiagsum[-1]

    # 寄与率の表示
    fig, ax = plt.subplots()
    # plt.plot(np.log10(sdiag),"o-")
    ax.plot(sdiag, ".-", label="contribution")
    ax.plot(sdiagsum, ".-", label="comulative contribution")
    ax.set_ylabel("rate")
    ax.set_xlabel("index")
    ax.legend()
    fig.tight_layout()
    fig.show()


'SVDでの寄与率'を見ておく。

In [ ]:
plot_svd_sdiag(g_df.values)


28次元程度で全寄与率がほぼ１になります。

In [ ]:
from sklearn.decomposition import NMF


def make_recom_svd(df, nrank):
    """line up candicates by SVD

    Args:
        df (pd.DataFrame): data
        nrank (int): the maximum rank to reconstruct data

    Returns:
        pd.DataFrame: reconstruct data
    """
    X = df.values
    u, sdiag, v = np.linalg.svd(X)
    print("sdiag", sdiag)
    s = np.zeros((u.shape[1], v.shape[0]))
    s[:nrank, :nrank] = np.diag(sdiag[:nrank])
    u = np.matrix(u)
    v = np.matrix(v)
    s = np.matrix(s)
    recom_svd = u * s * v
    return pd.DataFrame(recom_svd, index=df.index, columns=df.columns)


def make_recom_nmf(df, nrank):
    """line up candicates by NMF

    Args:
        df (pd.DataFrame): data
        nrank (int): the maximum rank to reconstruct data

    Returns:
        pd.DataFrame: reconstruct data
    """
    X = df.values
    model = NMF(n_components=nrank, init='random',
                shuffle=True, random_state=3)
    W = model.fit_transform(X)
    H = model.components_
    W = np.matrix(W)
    H = np.matrix(H)
    WH = W*H
    if False:
        """
        どの程度同じか調べる。
        """
        WHM = WH - X
        for i in range(WHM.shape[0]):
            for j in range(WHM.shape[1]):
                if np.abs(WHM[i, j]) > 0.1:
                    print(i, j, WHM[i, j])

    recom_nmf = WH
    return pd.DataFrame(recom_nmf, index=df.index, columns=df.columns)


def make_recom_correlation(df, nrank=None):
    """line up candicates by correlation

     X[material , structuretype]とすると
    ( X.T * X )[structuretype,structuretype] でstructuretype間の相関を与えるだろう。
    更にXをかけると[material, structuretype]の行列になる。
    recom = X[material , structuretype] * ( X.T * X )[structuretype,structuretype]
    Args:
        df (pd.DataFrame): data
        nrank (int): the maximum rank to reconstruct data

    Returns:
        pd.DataFrame: reconstruct data
    """
    # nrank はdummy
    X = np.matrix(df.values)
    """
    X[material , structuretype]とすると
    ( X.T * X )[structuretype,structuretype] でstructuretype間の相関を与えるだろう。
    更にXをかけると[material, structuretype]の行列になる。
    recom = X[material , structuretype] * ( X.T * X )[structuretype,structuretype]
    """
    recom = X * X.T * X
    # X^3のオーダーになっているので[0,1]に規格化する。
    vmax = recom.reshape(-1).max()
    vmin = recom.reshape(-1).min()
    recom = (recom - vmin)/(vmax-vmin)

    return pd.DataFrame(recom, index=df.index, columns=df.columns)


例えば、10次元分を考慮して低ランク行列を作ります。

In [ ]:
g_nrank = 10
print("nrank=", g_nrank)
g_df_recom = make_recom_svd(g_df, g_nrank)
plot_2df(g_df, g_df_recom, g_nrank)


左図で黒かった部分で、右図では色が淡い部分が存在する。
差をとり明瞭化する。

In [ ]:
plot_df_heatmap(g_df_recom-g_df, filename="image_executed/recommend.png")


### plotlyを用いた対話的可視化

次はplotlyがあると実行できる。

In [ ]:
import plotly.express as px
g_fig = px.imshow(g_df_recom-g_df)
g_fig.show()


明るい点（データインスタンス）が現れているのが分かります。これらの物質を、例えば差が0.35以上の物質に対して具体的に表示します。

In [ ]:
def print_existence(df, df_ref, threshold=0.35):
    """print the points the value of which is more than threshold

    Args:
        df (pd.DataFrame): data
        df_ref (pd.DataFrame): reference data
        threshold (float, optional): the threshold value. Defaults to 0.3.
    """
    df_ = df
    resultlist = []
    for name1 in df_.index:
        for name2 in df_.columns:
            value = df_.loc[name1, name2]
            exist_in_ref = df_ref.loc[name1, name2]
            if value >= threshold and exist_in_ref < 1:
                resultlist.append([name1, name2, value, exist_in_ref < 1])

    dfresult = pd.DataFrame(resultlist,
                            columns=["name1", "name2", "recom-ref", "not_exist_ref"])
    return dfresult.sort_values(by="recom-ref", ascending=False)


print_existence(g_df_recom-g_df, g_df, threshold=0.35)


この話の範囲内ではvalueが大きいほど可能性が高いことになります。
しかし、次元圧縮が教師なし学習なので実際は該当物質が何も見つからなくてもおかしくはありませんし、低ランク行列の次元や手法によりかなり結果は異なります。

念のため
atomwork (https://crystdb.nims.go.jp/)
で似た構造が無いかを確認すると

- SnAs
```
Sn3.6As3 R-3m
```
- InSb
```
In0.4Sb0.6 R-3m, Pm-3m 
In0.5Sb0.5 I41/amd
In0.7Sb0.3 P6/mmm
```

- PSi
```
Siの方が多い比率には存在無し。
```
- SnSb
```
Sn0.9Sb0.1 I41/amd
```

があります。

### モデルの作り方

低ランク行列の次元、低ランク行列の作り方などのパラメタがありますから、
教師あり学習の場合と同様に、既存観測データを訓練データとテストデータとに分けてパラメタチューニングすることが行われます。


#### 参考文献

以下の文献では高次元のテンソルのままの因子分解による次元圧縮も行っています。
高次元のテンソルでの分解による低ランク近似を用いたほうが二次元行列での低ランク近似よりも発見確率が高いという報告をしています。

1. "Matrix- and tensor-based recommender systems for the discovery of currently unknown inorganic compounds",
Atsuto Seko, Hiroyuki Hayashi, Hisashi Kashima, Isao Tanaka,
Phys. Rev. Materials 2, 013805 (2018), 
DOI: https://doi.org/10.1103/PhysRevMaterials.2.013805

2. COD
http://www.crystallography.net/cod/

3. atomwork
https://crystdb.nims.go.jp/
